<div style="background:linear-gradient(90deg,#1c3f6e 0%,#4d6d9a 45%,#b4636b 78%,#d97e5a 100%);
            padding:22px 26px;border-radius:10px;color:#fff;font-family:sans-serif">
<h1 style="margin:0 0 6px 0;font-size:30px">🦵 RSNA Knee Abnormality Detection</h1>
<h3 style="margin:0;font-weight:400;opacity:.92">A strong, runnable baseline — EDA → data model → report teacher → EfficientNetV2 2.5D student</h3>
</div>

**Task** — predict per-study probabilities for 12 knee abnormalities.
**Metric** — macro-averaged ROC-AUC over the 12 columns.
**Data** — ~570 GB of DICOM plus one free-text radiology report per study, in several languages.
Only a small fraction of the training studies carry hard labels.

---

## Five facts that dictate the design

**1. There are no reports at test time.** `test.csv` has a single column, `StudyInstanceUID`.
Text is therefore not a second input modality — it is the **label source**. The pipeline is:

```
Report ──teacher──▶ 12 soft labels for the whole train set ──▶ image-only student ──▶ submission
```

Any "image + text fusion" model simply cannot be assembled at inference.

**2. Knee laterality is decisive.** Four of the twelve targets
(`Medial/Lateral Meniscus`, `Medial/Lateral OA`) require telling the medial side from the lateral one.
Left and right knees are mirrored on coronal and axial planes, and on sagittal the slice order runs
medial→lateral in opposite directions. Without normalising every volume to a "right knee" the network
cannot separate those pairs even in principle.
**Corollary: horizontal flip is banned** in augmentation and TTA — it destroys medial/lateral.

**3. A study is a set of series with different roles.** Sagittal fluid-sensitive is the meniscus/ACL
view, coronal fat-sat carries MCL and the OA compartments, axial carries PF OA, synovitis and Baker's
cyst. The model should see several *view slots* per study and pool them with attention, with a mask for
missing slots.

**4. 570 GB cannot be decoded on the fly.** Training needs a cache of `uint8` volumes, produced in
shards. Inference (~1300 studies) is small enough to read DICOM directly.

**5. Macro-AUC is rank-based, per column.** No calibration, no thresholds. The prevalence shift between
train and leaderboard that the organisers warn about does not affect AUC at all.

## What this notebook changes versus the public EfficientNet 2.5D baseline

That notebook is a reasonable skeleton and its backbone choice is kept here. These are the issues that
stop it from training well — or from running at all.

| # | Issue in the public baseline | Consequence | Fix here |
|---|---|---|---|
| 1 | `timm.create_model('efficientnetv2_s')` | **Crashes** — that name is not registered in timm (the real ones are `tf_efficientnetv2_s`, `efficientnetv2_rw_s`) | `resolve_backbone()` picks the first name that actually exists |
| 2 | `pretrained=False` | Random init on a few thousand studies — the single largest score loss | `pretrained=True` when training; weights loaded from a dataset at submit time |
| 3 | `T.RandomHorizontalFlip(p=0.5)` | Destroys medial/lateral, i.e. 4 of the 12 targets | no flips; laterality normalised instead |
| 4 | `self.augment` is built but never called in `__getitem__` | No augmentation happens at all | augmentation applied explicitly to the volume |
| 5 | `sorted(folder.glob('*.dcm'))` | Sorts by SOP UID — an essentially random slice order | sorted by `ImagePositionPatient` projected on the slice normal |
| 6 | Only middle slices are read | Baker's cyst and MCL live at the periphery of the stack | slices sampled uniformly across the volume |
| 7 | Trains on labelled studies only | Discards most of the dataset | report teacher labels the rest |
| 8 | `StratifiedGroupKFold(..., groups=StudyInstanceUID)` on one label | Groups are unique, so grouping is a no-op; rare targets end up unbalanced | iterative multilabel stratification + grouping by duplicate report |
| 9 | `pos_weight = neg/pos` | Up to ~100× on rare labels — unstable gradients, and AUC ignores the prior anyway | plain BCE on soft targets |
| 10 | Every series of every study decoded per item | Dataloader-bound; one epoch takes hours | pre-cached `uint8` volumes |
| 11 | `MONOCHROME1`, `RescaleSlope`, anisotropic spacing ignored | Some studies come out inverted or stretched | handled inside the loader |
| 12 | Validation AUC over whatever labels are present | Silently optimistic | validation restricted to hard labels |

## 0. Configuration and theme

One switch block drives the whole notebook. On Kaggle it is convenient to split this into three
notebooks (`preprocess` → `train` → `submit`), but the code is identical.

In [ ]:
import os, re, sys, gc, json, math, time, warnings, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)


class CFG:
    # --- paths ---
    comp_dir    = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
    work_dir    = Path('/kaggle/working')
    cache_dir   = Path('/kaggle/temp/cache')     # preprocessing output
    extra_cache = []                             # finished shards attached as datasets

    # --- switches ---
    RUN_EDA        = True
    RUN_TEACHER    = True     # report -> labels
    RUN_PREPROCESS = False    # DICOM -> npz, slow, run in shards
    RUN_TRAIN      = False
    RUN_INFER      = False

    # --- preprocessing shards ---
    SHARD_ID   = 0
    NUM_SHARDS = 8

    # --- cache geometry ---
    IMG_SIZE = 224
    N_SLICES = 16
    VIEWS    = ['SAG_FS', 'COR_FS', 'AX_FS']
    N_VIEWS  = 3

    # --- model ---
    backbone   = ['tf_efficientnetv2_s.in21k_ft_in1k',   # tried in order, first available wins
                  'tf_efficientnetv2_s',
                  'efficientnetv2_rw_s',
                  'tf_efficientnetv2_b0']
    pretrained = True          # needs internet -> training notebook only
    drop_rate  = 0.3
    grad_ckpt  = True          # ~30% slower, much lower memory

    # --- training ---
    folds       = 5
    train_folds = [0]
    epochs      = 6
    batch_size  = 2            # studies, i.e. 2*3*16 = 96 images per forward
    accum       = 4
    lr          = 3e-4         # head
    backbone_lr = 1e-4
    wd          = 1e-2
    pseudo_w    = 0.5          # weight of report-derived soft labels vs hard labels
    aux_w       = 0.3          # per-view auxiliary head
    seed        = 42
    num_workers = 4
    amp         = True

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
           'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
           'Synovitis', "Baker's", 'Contusion', 'Fracture']

CFG.cache_dir.mkdir(parents=True, exist_ok=True)
CFG.work_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------ icefire theme
ICEFIRE  = sns.color_palette('icefire', 12)    # categorical ramp, cold -> hot
ICE      = ICEFIRE[2]                          # cold accent
FIRE     = ICEFIRE[9]                          # warm accent
MIDLINE  = '#39414d'
CMAP     = 'icefire'                                            # diverging: lift, centred maps
CMAP_SEQ = sns.blend_palette(['#eef1f6', ICE, FIRE], as_cmap=True)   # sequential, same family

sns.set_theme(style='white', palette=ICEFIRE, font_scale=0.95)
mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c8cdd4', 'axes.labelcolor': MIDLINE,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlepad': 10,
    'text.color': MIDLINE, 'xtick.color': MIDLINE, 'ytick.color': MIDLINE,
    'grid.color': '#e8eaee', 'axes.grid': True, 'axes.grid.axis': 'y',
    'legend.frameon': False, 'figure.facecolor': 'white',
})

def grad(n, reverse=False):
    """n colours along the icefire ramp, skipping its near-black centre.

    icefire is diverging: sampling it directly gives a muddy black bar in the middle of any
    categorical chart. Taking colours from the two lobes keeps the cold->hot reading without it.
    """
    cm = plt.get_cmap(CMAP)
    cold = np.linspace(0.02, 0.28, n - n // 2)      # icefire is near-black between ~0.35 and ~0.75
    warm = np.linspace(0.78, 0.99, n // 2)
    cols = [cm(p) for p in np.concatenate([cold, warm])][:n]
    return cols[::-1] if reverse else cols


def seed_everything(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

seed_everything(CFG.seed)

if not CFG.comp_dir.exists():                       # be forgiving about the mount point
    for cand in list(Path('/kaggle/input').glob('*knee*')) + \
                list(Path('/kaggle/input/competitions').glob('*knee*')):
        CFG.comp_dir = cand
        break
print('comp_dir :', CFG.comp_dir, '| exists:', CFG.comp_dir.exists())

### Environment check

Series arrive in a mix of transfer syntaxes, including JPEG 2000 and JPEG Lossless. `pydicom` cannot
decode those on its own — it needs `pylibjpeg` (+ `pylibjpeg-openjpeg`, `pylibjpeg-libjpeg`) or
`python-gdcm`. **The submission notebook has no internet**, so those wheels must be attached as a
dataset. Better to learn that here than through a wall of exceptions three hours in.

In [ ]:
import pydicom
print('pydicom :', pydicom.__version__)

decoders = {}
for mod in ['pylibjpeg', 'openjpeg', 'libjpeg', 'gdcm', 'PIL']:
    try:
        __import__(mod)
        decoders[mod] = 'ok'
    except Exception as e:
        decoders[mod] = f'MISSING ({type(e).__name__})'
for k, v in decoders.items():
    print(f'  {k:<10}: {v}')

if 'MISSING' in decoders['openjpeg'] and 'MISSING' in decoders['gdcm']:
    print('\n[!] No JPEG-2000 decoder found — compressed series will fail to decode.')
    print('    pip install pylibjpeg pylibjpeg-openjpeg pylibjpeg-libjpeg python-gdcm')
    print('    and attach the wheels as a dataset for the offline submission notebook.')

try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only'
    print(f'\ntorch   : {torch.__version__} | cuda: {torch.cuda.is_available()} | {gpu}')
except ImportError:
    print('\ntorch not available (EDA-only environment)')

In [ ]:
train        = pd.read_csv(CFG.comp_dir / 'train.csv')
train_series = pd.read_csv(CFG.comp_dir / 'train_series.csv')
test         = pd.read_csv(CFG.comp_dir / 'test.csv')
test_series  = pd.read_csv(CFG.comp_dir / 'test_series.csv')
sample_sub   = pd.read_csv(CFG.comp_dir / 'sample_submission.csv')

print(f'train studies : {len(train):,}')
print(f'train series  : {len(train_series):,}  ({len(train_series)/len(train):.2f} per study)')
print(f'test studies  : {len(test):,}  (about 1300 in the real test set)')
print('\ntrain columns :', train.columns.tolist())
print('test  columns :', test.columns.tolist())
assert 'Report' not in test.columns, \
    'If reports ever appear in the test set the whole strategy changes — check this.'
train.head(3)

# Part 1 — EDA

Not decoration: every plot below answers a question that changes the code further down.

## 1.1 How much of the data is actually labelled

In [ ]:
has_label  = train[TARGETS].notna().all(axis=1)
has_report = train['Report'].fillna('').str.strip().str.len() > 0

print(f'hard labels        : {has_label.sum():,} / {len(train):,}  ({has_label.mean()*100:.1f}%)')
print(f'report present     : {has_report.sum():,} / {len(train):,}  ({has_report.mean()*100:.1f}%)')
print(f'neither            : {(~has_label & ~has_report).sum():,}  <- unusable for supervision')
print(f'partially labelled : {(train[TARGETS].notna().any(axis=1) & ~has_label).sum():,}')

lab  = train[has_label].copy()
prev = lab[TARGETS].mean().sort_values(ascending=False)
summary = pd.DataFrame({'pos_rate': prev,
                        'n_pos': lab[TARGETS].sum().reindex(prev.index).astype(int)})
summary['n_pos_per_val_fold'] = (summary['n_pos'] / CFG.folds).round().astype(int)
print('\nPrevalence on the labelled subset:')
display(summary.style.background_gradient(cmap=CMAP_SEQ, subset=['pos_rate'])
                     .format({'pos_rate': '{:.3f}'}))
print('Watch n_pos_per_val_fold: below ~30 the single-fold AUC of that target swings by +-0.05.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))

ax = axes[0]
ax.barh(prev.index[::-1], prev.values[::-1] * 100, color=grad(12), edgecolor='white', height=.75)
ax.set_xlabel('positive rate, %'); ax.set_title('Prevalence of each target (hard labels)')
ax.grid(axis='x'); ax.grid(axis='y', visible=False)
for i, v in enumerate(prev.values[::-1]):
    ax.text(v * 100 + .35, i, f'{v*100:.1f}', va='center', fontsize=8, color=MIDLINE)

ax = axes[1]
npl = lab[TARGETS].sum(axis=1)
vals, cnts = np.unique(npl, return_counts=True)
ax.bar(vals, cnts, color=grad(len(vals)), edgecolor='white')
ax.set_xlabel('abnormalities per study'); ax.set_ylabel('studies')
ax.set_title(f'Multi-label density (mean = {npl.mean():.2f})')
plt.tight_layout(); plt.show()

## 1.2 Co-occurrence — use lift, not conditional probability

`P(col | row)` mostly reflects the prevalence of the column, so it looks dramatic without saying much.
Lift, `P(A,B) / P(A)P(B)`, isolates the actual association. Pairs well above 1 (typically
`Medial OA` ↔ `Lateral OA`, `ACL` ↔ `Contusion`) will be learnt jointly by a shared trunk; isolated
targets such as `Fracture` need their own attention.

In [ ]:
P = lab[TARGETS].mean().values
J = (lab[TARGETS].T.values @ lab[TARGETS].values) / len(lab)
lift = pd.DataFrame(J / (P[:, None] * P[None, :] + 1e-9), index=TARGETS, columns=TARGETS)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(lift, annot=True, fmt='.1f', cmap=CMAP, center=1.0, vmin=0, vmax=3,
            mask=np.eye(12, dtype=bool), linewidths=.6, linecolor='white', ax=ax,
            annot_kws={'size': 8}, cbar_kws={'label': 'lift = P(A,B) / P(A)P(B)', 'shrink': .8})
ax.set_title('Co-occurrence lift — above 1 means the pair appears together more often than chance')
plt.tight_layout(); plt.show()

## 1.3 Series inventory — which views a study actually has

The question that decides the architecture: **for how many studies does each
(plane × fluid-sensitive × fat-sat) combination exist?** If axial is present for only 60% of studies,
the model must handle a missing view through masking rather than crash or read zeros as signal.

In [ ]:
def view_key(r):
    plane = str(r['Anatomical_Plane'])[:3].upper()
    return f"{plane}_{'FS' if r['Fluid_Sensitive'] == 1 else 'T1'}{'_FAT' if r['Fat_Suppression'] == 1 else ''}"

ts = train_series.copy()
ts['view'] = ts.apply(view_key, axis=1)

combo = (ts.groupby(['Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression'])
           .size().reset_index(name='n_series').sort_values('n_series', ascending=False))
combo['pct'] = (combo.n_series / len(ts) * 100).round(1)
print('Series type combinations:'); display(combo)

cov = ts.assign(one=1).pivot_table(index='StudyInstanceUID', columns='view',
                                   values='one', aggfunc='sum', fill_value=0)
coverage = pd.DataFrame({
    'studies_with_view': (cov > 0).sum(),
    'pct_studies': ((cov > 0).mean() * 100).round(1),
    'mean_series_when_present': cov.replace(0, np.nan).mean().round(2),
}).sort_values('pct_studies', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.4))
ax = axes[0]
ax.barh(coverage.index[::-1], coverage.pct_studies.values[::-1],
        color=grad(len(coverage)), edgecolor='white', height=.7)
ax.axvline(80, color=FIRE, ls='--', lw=1.2, label='80% — below this, make the view optional')
ax.set_xlabel('% of studies containing the view'); ax.set_title('View coverage per study')
ax.grid(axis='x'); ax.grid(axis='y', visible=False); ax.legend(fontsize=8)

ax = axes[1]
plane_cov = (ts.pivot_table(index='StudyInstanceUID', columns='Anatomical_Plane',
                            values='Fluid_Sensitive', aggfunc='size', fill_value=0) > 0)
pc = (plane_cov.mean() * 100).sort_values(ascending=False)
ax.bar(pc.index, pc.values, color=grad(len(pc)), edgecolor='white', width=.55)
ax.set_ylim(0, 108); ax.set_ylabel('% of studies'); ax.set_title('Coverage by anatomical plane')
for i, v in enumerate(pc.values):
    ax.text(i, v + 1.8, f'{v:.1f}', ha='center', fontsize=9, color=MIDLINE)
plt.tight_layout(); plt.show()

display(coverage)
print(f'studies with all three planes           : {plane_cov.all(axis=1).mean()*100:.1f}%')
print(f"studies with any fluid-sensitive series : "
      f"{ts.groupby('StudyInstanceUID')['Fluid_Sensitive'].max().mean()*100:.1f}%")

## 1.4 DICOM audit — tags, transfer syntax, laterality

The organisers kept 86 tags. Which ones matters a great deal: can we read the knee side
(`Laterality` / `ImageLaterality`), can we sort slices geometrically (`ImagePositionPatient` +
`ImageOrientationPatient`), is `PatientID` present (fold leakage if one patient has several studies)?

In [ ]:
def sample_dcm_paths(root, n_series=60, per_series=2, seed=0):
    rng = np.random.default_rng(seed)
    series_dirs = [p for p in root.glob('*/*') if p.is_dir()]
    if not series_dirs:
        return []
    pick = rng.choice(len(series_dirs), size=min(n_series, len(series_dirs)), replace=False)
    return [f for i in pick for f in sorted(series_dirs[i].glob('*.dcm'))[:per_series]]


paths = sample_dcm_paths(CFG.comp_dir / 'train_series')
print('scanning files:', len(paths))

tag_counter, rows = Counter(), []
for p in paths:
    try:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
    except Exception as e:
        print('read failed:', p.name, e); continue
    for el in ds:
        tag_counter[el.keyword or str(el.tag)] += 1
    rows.append({
        'Rows': getattr(ds, 'Rows', None), 'Columns': getattr(ds, 'Columns', None),
        'PixelSpacing': str(getattr(ds, 'PixelSpacing', None)),
        'SliceThickness': getattr(ds, 'SliceThickness', None),
        'Photometric': getattr(ds, 'PhotometricInterpretation', None),
        'Laterality': getattr(ds, 'Laterality', None) or getattr(ds, 'ImageLaterality', None),
        'SeriesDescription': getattr(ds, 'SeriesDescription', None),
        'FieldStrength': getattr(ds, 'MagneticFieldStrength', None),
        'Manufacturer': getattr(ds, 'Manufacturer', None),
        'TransferSyntax': str(ds.file_meta.TransferSyntaxUID) if hasattr(ds, 'file_meta') else None,
        'has_IPP': hasattr(ds, 'ImagePositionPatient'),
        'has_IOP': hasattr(ds, 'ImageOrientationPatient'),
        'has_PatientID': hasattr(ds, 'PatientID'),
    })

meta = pd.DataFrame(rows)
print('\n=== available tags, top 40 by frequency ===')
print(pd.Series(tag_counter).sort_values(ascending=False).head(40).to_string())
for c in ['Laterality', 'Photometric', 'TransferSyntax', 'Manufacturer',
          'has_IPP', 'has_IOP', 'has_PatientID']:
    print(f'\n--- {c} ---')
    print(meta[c].value_counts(dropna=False).head(8).to_string())
print('\n--- geometry ---')
print(meta[['Rows', 'Columns', 'SliceThickness']].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col, title in zip(axes, ['Laterality', 'Photometric', 'Manufacturer'],
                          ['Knee side', 'Photometric interpretation', 'Scanner vendor']):
    vc = meta[col].fillna('missing').value_counts().head(6)
    ax.bar(vc.index.astype(str), vc.values, color=grad(max(len(vc), 3)), edgecolor='white', width=.6)
    ax.set_title(title); ax.tick_params(axis='x', rotation=20, labelsize=8)
plt.tight_layout(); plt.show()

**Audit checklist**

- `Laterality` / `ImageLaterality` populated → the knee side can be normalised directly. If empty,
  fall back to the sign of X in `ImagePositionPatient` (in the LPS frame a right knee sits at x < 0).
- `MONOCHROME1` present → those slices must be inverted.
- Several transfer syntaxes → the JPEG-2000 decoders flagged above are mandatory, not optional.
- `PatientID` present → group the folds by patient.

## 1.5 Leakage check

In [ ]:
if meta['has_PatientID'].any():
    print('PatientID exists in the headers -> collect it during preprocessing and group folds by it.')
else:
    print('PatientID is not among the retained tags.')

rep = train['Report'].fillna('').str.strip()
long_rep = rep[rep.str.len() > 30]
dup = long_rep.duplicated(keep=False)
print(f'\nexactly duplicated non-empty reports: {dup.sum():,} rows '
      f'({long_rep[dup].nunique():,} unique texts)')
if dup.sum():
    print('-> those studies must share a fold; handled in Part 4 via a report hash group.')

## 1.6 Reports — length, language, keyword signal

Hand-written stop-word heuristics make a poor language detector: `la ` occurs in French, Spanish and
Italian alike, and an `if/elif` chain bakes in an ordering bias. The estimate below is at least
symmetric, and it is only used for orientation — the teacher in Part 3 relies on **character n-grams,
which are language-agnostic by construction**.

In [ ]:
rep_len = train['Report'].fillna('').str.split().str.len()
print(rep_len.describe().round(1).to_string())

LANG_HINTS = {
    'en': [' the ', ' and ', ' with ', ' there is ', ' no evidence '],
    'nl': [' de ', ' het ', ' en ', ' geen ', ' voorste ', ' kruisband '],
    'de': [' der ', ' die ', ' das ', ' und ', ' kein ', ' kreuzband '],
    'fr': [' le ', ' la ', ' des ', ' avec ', ' pas de ', ' croise '],
    'es': [' el ', ' los ', ' con ', ' sin ', ' rodilla ', ' menisco '],
    'pt': [' do ', ' da ', ' com ', ' sem ', ' joelho ', ' menisco '],
    'it': [' il ', ' della ', ' con ', ' senza ', ' ginocchio ', ' menisco '],
    'tr': [' ve ', ' ile ', ' yok', ' diz ', ' menisk'],
}

def guess_lang(t):
    if not isinstance(t, str) or len(t.strip()) < 15:
        return 'empty'
    s = ' ' + t.lower() + ' '
    scores = {k: sum(s.count(w) for w in v) for k, v in LANG_HINTS.items()}
    best, val = max(scores.items(), key=lambda kv: kv[1])
    return best if val > 0 else 'other'

train['lang_guess'] = train['Report'].apply(guess_lang)
lc = train['lang_guess'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
ax = axes[0]
ax.hist(rep_len.clip(0, rep_len.quantile(.99)), bins=50, color=ICE, edgecolor='white')
ax.axvline(rep_len.median(), color=FIRE, ls='--', lw=1.4,
           label=f'median = {rep_len.median():.0f} words')
ax.set_xlabel('report length, words'); ax.set_ylabel('studies')
ax.set_title('Report length'); ax.legend(fontsize=8)

ax = axes[1]
ax.bar(lc.index, lc.values, color=grad(max(len(lc), 3)), edgecolor='white', width=.6)
ax.set_title('Estimated report language (rough)'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

print('\n=== sample report ===')
if (has_label & has_report).any():
    print(train.loc[has_label & has_report, 'Report'].iloc[0][:800])

## 1.7 Visual check — slices and laterality

One study across all of its series. This is also where the laterality story gets confirmed by eye: on a
coronal slice the medial compartment of a right knee and of a left knee sit on opposite sides of the frame.

In [ ]:
def read_pixels(path):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    img = img * float(getattr(ds, 'RescaleSlope', 1) or 1) + \
          float(getattr(ds, 'RescaleIntercept', 0) or 0)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        img = img.max() - img
    lo, hi = np.percentile(img, [0.5, 99.5])
    return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1), ds


demo_study = train_series['StudyInstanceUID'].iloc[0]
sers = train_series[train_series.StudyInstanceUID == demo_study]
fig, axes = plt.subplots(len(sers), 5, figsize=(15, 3 * len(sers)))
axes = np.atleast_2d(axes)
for r, (_, s) in enumerate(sers.iterrows()):
    files = sorted((CFG.comp_dir / 'train_series' / demo_study / s.SeriesInstanceUID).glob('*.dcm'))
    idxs = np.linspace(0, len(files) - 1, 5).astype(int) if files else []
    for c, i in enumerate(idxs):
        try:
            img, ds = read_pixels(files[i])
            axes[r, c].imshow(img, cmap='gray')
            if c == 0:
                side = getattr(ds, 'Laterality', None) or getattr(ds, 'ImageLaterality', '?')
                axes[r, c].set_title(
                    f'{s.Anatomical_Plane} | FS={s.Fluid_Sensitive} | FAT={s.Fat_Suppression} | side={side}',
                    fontsize=8, loc='left', color=FIRE)
        except Exception as e:
            axes[r, c].text(.5, .5, str(e)[:32], ha='center', fontsize=6)
        axes[r, c].axis('off'); axes[r, c].grid(False)
plt.suptitle(f'study {demo_study[:26]}...', y=1.0, fontweight='bold')
plt.tight_layout(); plt.show()

# Part 2 — Data model

Three entities and one policy.

```
Study ──< Series ──< Slice
  │          │
  │          └── plane, fluid_sensitive, fat_sat, laterality, n_slices
  └── VIEW SLOTS:  SAG_FS · COR_FS · AX_FS   (+ presence mask)
```

**Series selection.** One series per slot:

| Slot | Preferred series | Targets it carries |
|---|---|---|
| `SAG_FS` | sagittal fluid-sensitive, fat-sat preferred | ACL, menisci, contusion |
| `COR_FS` | coronal fluid-sensitive, fat-sat preferred | MCL, OA compartments, contusion |
| `AX_FS`  | axial fluid-sensitive | PF OA, synovitis, Baker's cyst, effusion |

Fallback cascade: exact match → same plane without fat-sat → same plane, anything → slot marked empty
(mask = 0) and ignored by the model. Ties are broken by slice count.

**Laterality normalisation.** Every volume is mapped onto a right knee: coronal and axial get their X
axis mirrored when `Laterality == 'L'`; sagittal gets its slice order reversed, because that is its
medial↔lateral axis. Only after this can `Medial *` and `Lateral *` be learnt at all.

**Cache format.** One `npz` per study: `uint8 (V, D, H, W)` plus a `(V,)` mask. At `V=3, D=16, H=W=224`
that is roughly 2.4 MB per study.

In [ ]:
VIEW_SPEC = {                       # slot -> (plane, fluid_sensitive, preferred fat_sat)
    'SAG_FS': ('Sagittal', 1, 1),
    'COR_FS': ('Coronal',  1, 1),
    'AX_FS':  ('Axial',    1, 0),
}

def assign_views(series_df: pd.DataFrame) -> dict:
    """For a single study, return {slot: SeriesInstanceUID or None}."""
    out = {}
    for slot, (plane, fluid, prefer_fat) in VIEW_SPEC.items():
        cand = series_df[series_df.Anatomical_Plane == plane]
        if len(cand) == 0:
            out[slot] = None
            continue
        tiers = [cand[(cand.Fluid_Sensitive == fluid) & (cand.Fat_Suppression == prefer_fat)],
                 cand[cand.Fluid_Sensitive == fluid],
                 cand]
        chosen = next((t for t in tiers if len(t) > 0), None)
        if chosen is None:
            out[slot] = None
            continue
        if 'n_files' in chosen.columns:
            chosen = chosen.sort_values('n_files', ascending=False)
        out[slot] = chosen.iloc[0].SeriesInstanceUID
    return out


def build_manifest(series_df: pd.DataFrame, root: Path, count_files=False) -> pd.DataFrame:
    df = series_df.copy()
    if count_files:
        df['n_files'] = [len(list((root / r.StudyInstanceUID / r.SeriesInstanceUID).glob('*.dcm')))
                         for r in df.itertuples()]
    rows = []
    for study, g in df.groupby('StudyInstanceUID'):
        rec = {'StudyInstanceUID': study}
        rec.update(assign_views(g))
        rec['n_series'] = len(g)
        rows.append(rec)
    return pd.DataFrame(rows)


# --- self-test on a synthetic study: sagittal fat-sat wins, axial is absent ---
_demo = pd.DataFrame([
    dict(SeriesInstanceUID='a', Anatomical_Plane='Sagittal', Fluid_Sensitive=1, Fat_Suppression=1),
    dict(SeriesInstanceUID='b', Anatomical_Plane='Sagittal', Fluid_Sensitive=0, Fat_Suppression=0),
    dict(SeriesInstanceUID='c', Anatomical_Plane='Coronal',  Fluid_Sensitive=1, Fat_Suppression=0),
])
assert assign_views(_demo) == {'SAG_FS': 'a', 'COR_FS': 'c', 'AX_FS': None}
print('assign_views self-test passed')

manifest = build_manifest(train_series, CFG.comp_dir / 'train_series')
fill = (manifest[list(VIEW_SPEC)].notna().mean() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(fill.index, fill.values, color=grad(3), edgecolor='white', width=.5)
ax.set_ylim(0, 112); ax.set_ylabel('% of studies'); ax.set_title('View slot fill rate')
for i, v in enumerate(fill.values):
    ax.text(i, v + 2.5, f'{v:.1f}', ha='center', fontsize=9, color=MIDLINE)
plt.tight_layout(); plt.show()

print(f"studies with no slot at all : {(manifest[list(VIEW_SPEC)].isna().all(axis=1)).sum()}")
print(f"studies with all three      : {(manifest[list(VIEW_SPEC)].notna().all(axis=1)).mean()*100:.1f}%")
manifest.head()

## 2.1 Preprocessing — DICOM to `uint8` volume

The details that break a naive implementation:

1. **Slice order.** `sorted(glob('*.dcm'))` sorts by SOP UID, which is effectively random. Sort by
   `ImagePositionPatient` projected onto the slice normal (the cross product of the two
   `ImageOrientationPatient` vectors), falling back to `InstanceNumber`.
2. **`MONOCHROME1`** must be inverted.
3. **Normalisation** is robust and per volume — MR intensities are not in Hounsfield units and are not
   comparable across scanners.
4. **Anisotropic pixels** are corrected via `PixelSpacing` before the square resize.
5. **Slice sampling** spans the whole volume rather than the middle window: Baker's cyst and MCL live
   at the periphery.
6. **Laterality** is normalised to a right knee.

Headers are read first with `stop_before_pixels=True`, so only the slices actually kept get decoded —
on a 300-slice series that alone is a large saving.

In [ ]:
import cv2
cv2.setNumThreads(0)          # essential: OpenCV threads deadlock inside DataLoader workers


def _slice_order(dss):
    try:
        iop = np.array(dss[0].ImageOrientationPatient, dtype=float)
        n = np.cross(iop[:3], iop[3:])
        pos = [float(np.dot(np.array(d.ImagePositionPatient, dtype=float), n)) for d in dss]
    except Exception:
        pos = [float(getattr(d, 'InstanceNumber', i) or i) for i, d in enumerate(dss)]
    return np.argsort(pos)


def _laterality(dss):
    for d in dss:
        v = getattr(d, 'Laterality', None) or getattr(d, 'ImageLaterality', None)
        if v in ('L', 'R'):
            return v
    try:                                     # LPS frame: a right knee sits at x < 0
        return 'R' if float(dss[0].ImagePositionPatient[0]) < 0 else 'L'
    except Exception:
        return 'R'


def load_series_volume(series_dir: Path, plane: str,
                       size=CFG.IMG_SIZE, n_slices=CFG.N_SLICES):
    """DICOM series -> uint8 array (n_slices, size, size), oriented as a right knee."""
    files = list(series_dir.glob('*.dcm'))
    if not files:
        return None

    heads = []
    for f in files:
        try:
            heads.append((f, pydicom.dcmread(str(f), stop_before_pixels=True)))
        except Exception:
            continue
    if not heads:
        return None

    order = _slice_order([h[1] for h in heads])
    lat = _laterality([h[1] for h in heads])
    take = np.linspace(0, len(order) - 1, min(n_slices, len(order))).round().astype(int)
    sel_files = [heads[order[i]][0] for i in take]

    ps = getattr(heads[0][1], 'PixelSpacing', None)
    sy, sx = (float(ps[0]), float(ps[1])) if ps else (1.0, 1.0)

    frames = []
    for f in sel_files:
        try:
            ds = pydicom.dcmread(str(f))
            a = ds.pixel_array.astype(np.float32)
        except Exception:
            continue
        a = a * float(getattr(ds, 'RescaleSlope', 1) or 1) + \
            float(getattr(ds, 'RescaleIntercept', 0) or 0)
        if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
            a = a.max() - a
        if a.ndim == 3:                                   # multiframe or RGB
            a = a[..., 0] if a.shape[-1] <= 4 else a[0]
        if abs(sy - sx) > 1e-3:
            a = cv2.resize(a, None, fx=sx / min(sx, sy), fy=sy / min(sx, sy),
                           interpolation=cv2.INTER_LINEAR)
        frames.append(cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA))
    if not frames:
        return None

    vol = np.stack(frames)
    lo, hi = np.percentile(vol, [0.5, 99.5])
    vol = (np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)

    if lat == 'L':                                        # normalise to a right knee
        vol = vol[:, :, ::-1] if plane in ('Coronal', 'Axial') else vol[::-1]
    vol = np.ascontiguousarray(vol)

    if vol.shape[0] < n_slices:                           # pad by repeating the last slice
        vol = np.concatenate([vol, np.repeat(vol[-1:], n_slices - vol.shape[0], axis=0)], 0)
    return vol


def cache_study(study: str, mrow, root: Path, out_dir: Path) -> bool:
    dst = out_dir / f'{study}.npz'
    if dst.exists():
        return True
    vols = np.zeros((CFG.N_VIEWS, CFG.N_SLICES, CFG.IMG_SIZE, CFG.IMG_SIZE), np.uint8)
    mask = np.zeros(CFG.N_VIEWS, np.uint8)
    for vi, slot in enumerate(CFG.VIEWS):
        sid = mrow.get(slot)
        if not isinstance(sid, str):
            continue
        v = load_series_volume(root / study / sid, VIEW_SPEC[slot][0])
        if v is not None:
            vols[vi], mask[vi] = v, 1
    if mask.sum() == 0:
        return False
    np.savez_compressed(dst, vol=vols, mask=mask)
    return True

print('preprocessing helpers ready')

### Preprocessing smoke test

Cache exactly one study and look at it before committing hours of compute. If the montage is upside
down, inverted or scrambled, fix it now rather than after eight shards.

In [ ]:
_row = manifest.iloc[0]
_t0 = time.time()
_ok = cache_study(_row.StudyInstanceUID, _row, CFG.comp_dir / 'train_series', CFG.cache_dir)
_dt = time.time() - _t0
print(f'cached: {_ok} in {_dt:.1f}s -> roughly {len(manifest)*_dt/3600/CFG.NUM_SHARDS:.1f} h per shard')

if _ok:
    z = np.load(CFG.cache_dir / f'{_row.StudyInstanceUID}.npz')
    vol, msk = z['vol'], z['mask']
    print('volume:', vol.shape, vol.dtype, '| mask:', msk,
          '| intensity mean/std:', round(float(vol.mean()), 1), round(float(vol.std()), 1))
    show = np.linspace(0, CFG.N_SLICES - 1, 8).astype(int)
    fig, axes = plt.subplots(CFG.N_VIEWS, 8, figsize=(15, 2.1 * CFG.N_VIEWS))
    for v in range(CFG.N_VIEWS):
        for c, s in enumerate(show):
            axes[v, c].imshow(vol[v, s], cmap='gray', vmin=0, vmax=255)
            axes[v, c].axis('off'); axes[v, c].grid(False)
            if c == 0:
                axes[v, c].set_title(f'{CFG.VIEWS[v]} (present={msk[v]})',
                                     loc='left', fontsize=9, color=FIRE)
    plt.suptitle('Cached study — every volume normalised to a right knee', y=1.01, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
if CFG.RUN_PREPROCESS:
    out_dir = CFG.cache_dir / f'shard{CFG.SHARD_ID}'
    out_dir.mkdir(parents=True, exist_ok=True)
    todo = manifest.iloc[CFG.SHARD_ID::CFG.NUM_SHARDS].reset_index(drop=True)
    print(f'shard {CFG.SHARD_ID}/{CFG.NUM_SHARDS}: {len(todo)} studies -> {out_dir}')

    t0, ok = time.time(), 0
    root = CFG.comp_dir / 'train_series'
    for i, r in todo.iterrows():
        try:
            ok += cache_study(r.StudyInstanceUID, r, root, out_dir)
        except Exception as e:
            print('failed', r.StudyInstanceUID[:16], type(e).__name__, e)
        if (i + 1) % 100 == 0:
            el = time.time() - t0
            print(f'{i+1}/{len(todo)} | ok={ok} | {el/60:.1f} min | '
                  f'ETA {(el/(i+1)*(len(todo)-i-1))/60:.1f} min', flush=True)
    print('done:', ok)
    # then: Save Version -> publish the output as a Kaggle Dataset -> list it in CFG.extra_cache

> **In practice:** run this notebook `NUM_SHARDS` times, changing `SHARD_ID` each time, publish every
> output as its own dataset, and attach all of them to the training notebook. Inference needs no cache —
> ~1300 studies are read straight from DICOM inside the DataLoader workers.

# Part 3 — Teacher: report → 12 labels

This is where most of the baseline's edge comes from. Hard labels are scarce, reports are nearly
universal, and the teacher turns the second into the first.

Two components:

1. **A multilingual lexicon with negation scope.** Interpretable, works on languages that barely appear
   in the labelled subset, and catches "ACL intact" or "pas de rupture".
2. **Character n-gram TF-IDF (3–5) plus logistic regression per label.** Character n-grams are
   language-agnostic and pick up morphology on their own: `menisc`, `kruisband`, `Kreuzband` and
   `menisküs` all become features without any translation step.

The final soft label mixes both, and a hard label always wins where it exists. The teacher is scored
honestly by out-of-fold AUC on the labelled subset: a target at 0.95+ produces trustworthy
pseudo-labels, one at 0.70 needs its weight cut.

In [ ]:
LEXICON = {
    'ACL': dict(
        pos=[r'\bacl\b', r'anterior cruciate', r'ligament(?:um)? crois[eé] ant[eé]rieur', r'\blca\b',
             r'vorder\w* kreuzband', r'voorste kruisband', r'\bvkb\b',
             r'ligamento cruzado anterior', r'legamento crociato anteriore',
             r'ön çapraz bağ', r'\böçb\b'],
        cond=[r'tear|rupture|torn|ruptu\w*|scheur|riss|rotura|rottura|lesi[oó]n|d[eé]chirure|yırtık|kopma']),
    'MCL': dict(
        pos=[r'\bmcl\b', r'medial collateral', r'ligament(?:um)? lat[eé]ral interne',
             r'inner\w* seitenband', r'mediale?s? kollateralband', r'mediale collaterale band',
             r'ligamento colateral medial', r'legamento collaterale mediale', r'medial kollateral'],
        cond=[r'tear|rupture|torn|sprain|ruptu\w*|riss|distors|lesi[oó]n|d[eé]chirure|yırtık']),
    'Medial Meniscus': dict(
        pos=[r'medial menisc\w+', r'm[eé]nisque interne', r'innenmeniskus', r'mediale meniscus',
             r'menisco (?:medial|interno)', r'iç menisküs', r'\bmm\b'],
        cond=[r'tear|torn|ruptu\w*|riss|scheur|rotura|lesi[oó]n|d[eé]chirure|yırtık|degenerat']),
    'Lateral Meniscus': dict(
        pos=[r'lateral menisc\w+', r'm[eé]nisque externe', r'aussenmeniskus|außenmeniskus',
             r'laterale meniscus', r'menisco (?:lateral|externo)', r'dış menisküs', r'\blm\b'],
        cond=[r'tear|torn|ruptu\w*|riss|scheur|rotura|lesi[oó]n|d[eé]chirure|yırtık|degenerat']),
    'Medial OA': dict(
        pos=[r'medial (?:compartment|tibiofemoral|femorotibial)', r'compartiment interne',
             r'mediale?n? (?:kompartiment|gelenkspalt)', r'mediale compartiment',
             r'compartimento medial'],
        cond=[r'osteoarthrit|arthros|gonarthros|arthrose|artrosis|artrose|cartilage loss|'
              r'chondropath|chondral|knorpel|kraakbeen|kıkırdak|osteofit|osteophyt|joint space narrow']),
    'Lateral OA': dict(
        pos=[r'lateral (?:compartment|tibiofemoral|femorotibial)', r'compartiment externe',
             r'lateral\w* (?:kompartiment|gelenkspalt)', r'laterale compartiment',
             r'compartimento lateral'],
        cond=[r'osteoarthrit|arthros|gonarthros|arthrose|artrosis|artrose|cartilage loss|'
              r'chondropath|chondral|knorpel|kraakbeen|kıkırdak|osteofit|osteophyt|joint space narrow']),
    'PF OA': dict(
        pos=[r'patellofemoral', r'f[eé]moro-?patellaire', r'retropatell\w+', r'patellofemoraal',
             r'patelofemoral', r'femoropatelar', r'patellar cartilage', r'patella\w* knorpel'],
        cond=[r'osteoarthrit|arthros|arthrose|artrosis|chondropath|chondral|chondromalac|'
              r'cartilage|knorpel|kraakbeen|osteophyt|osteofit']),
    'Effusion': dict(
        pos=[r'effusion', r'joint fluid', r'[ée]panchement', r'gelenkerguss|erguss', r'gewrichtsvocht',
             r'derrame(?: articular)?', r'versamento', r'eklem s[ıi]v[ıi]s[ıi]|efüzyon', r'hydrops'],
        cond=[]),
    'Synovitis': dict(
        pos=[r'synovit\w+', r'synovial (?:thickening|proliferation|hypertroph)', r'synovialitis',
             r'sinovitis', r'sinovite', r'synovite', r'sinovit'],
        cond=[]),
    "Baker's": dict(
        pos=[r"baker'?s? cyst", r'popliteal cyst', r'kyste (?:de )?baker|kyste poplit[eé]',
             r'baker[- ]?zyste|poplitealzyste', r'bakercyste', r'quiste de baker',
             r'cisti di baker', r'baker kisti', r'gastrocnemio-?semimembranosus'],
        cond=[]),
    'Contusion': dict(
        pos=[r'contusion', r'bone bruise', r'bone marrow (?:edema|oedema)', r'knochenmark[soö]dem',
             r'beenmerg[oö]edeem', r'[oó]edema (?:[oó]seo|de m[eé]dula)', r'contus[aã]o',
             r'contusione', r'kemik ili[gğ]i [oö]dem', r'trabecular (?:edema|injury)'],
        cond=[]),
    'Fracture': dict(
        pos=[r'fracture', r'fraktur', r'fractuur', r'fractura', r'frattura', r'k[ıi]r[ıi][kg]',
             r'avulsion', r'segond'],
        cond=[]),
}

NEG_CUES = [
    r'\bno\b', r'\bnot\b', r'\bwithout\b', r'\bintact\b', r'\bnormal\w*', r'\bunremarkable\b',
    r'\bnegative\b', r'\bfree of\b', r'\bruled out\b', r'\bgeen\b', r'\bkein\w*', r'\bohne\b',
    r'\bintakt\b', r"\bpas d[eu']", r'\bsans\b', r'\baucun\w*', r'\babsence\b', r'\bsin\b',
    r'\bsem\b', r'\bsenza\b', r'\bnessun\w*', r'\byok(?:tur)?\b', r'\bdo[gğ]al\b',
    r'\bregelrecht\b', r'\bunauff[aä]llig\b', r'\bnegatif\b',
]
NEG_RE = re.compile('|'.join(NEG_CUES), re.I)
CLAUSE_SPLIT = re.compile(r'[.;:\n]')


def rule_score(text: str) -> dict:
    """Rule-based estimate per label: 0.03 / 0.10 / 0.15 / 0.90.

    Key detail: negation is searched only inside the current clause. Otherwise
    "Kein Erguss. Innenmeniskus Riss" marks the meniscal tear as negated.
    """
    if not isinstance(text, str) or not text.strip():
        return {t: np.nan for t in TARGETS}
    t = ' ' + re.sub(r'\s+', ' ', text.lower()) + ' '
    out = {}
    for label, spec in LEXICON.items():
        pos_re = re.compile('|'.join(spec['pos']), re.I)
        cond_re = re.compile('|'.join(spec['cond']), re.I) if spec['cond'] else None
        hits = negated = conditioned = 0
        for m in pos_re.finditer(t):
            hits += 1
            left = CLAUSE_SPLIT.split(t[max(0, m.start() - 80):m.start()])[-1]
            right = CLAUSE_SPLIT.split(t[m.end():m.end() + 110])[0]
            if NEG_RE.search(left) or NEG_RE.search(right[:40]):
                negated += 1
            if cond_re is None or cond_re.search(left) or cond_re.search(right):
                conditioned += 1
        if hits == 0:
            out[label] = 0.03                       # never mentioned
        elif conditioned == 0:
            out[label] = 0.15                       # mentioned, no finding word nearby
        elif negated >= conditioned:
            out[label] = 0.10                       # explicitly negated
        else:
            out[label] = 0.90                       # asserted
    return out

In [ ]:
# --- self-test: rerun this after every edit to the lexicon ---
SANITY = {
    'EN pos': 'Complete tear of the anterior cruciate ligament. Medial meniscus posterior horn tear. '
              'Small joint effusion.',
    'EN neg': 'The anterior cruciate ligament is intact. No meniscal tear. No joint effusion.',
    'NL pos': 'Ruptuur van de voorste kruisband. Scheur mediale meniscus achterhoorn. Bakercyste aanwezig.',
    'DE mix': 'Vorderes Kreuzband intakt. Kein Erguss. Innenmeniskus Riss im Hinterhorn.',
    'FR mix': "Rupture du ligament croisé antérieur. Pas d'épanchement articulaire. Kyste de Baker.",
    'ES pos': 'Rotura del menisco medial. Derrame articular moderado. Artrosis del compartimento medial.',
}
_s = pd.DataFrame({k: rule_score(v) for k, v in SANITY.items()}).T[TARGETS]

assert _s.loc['EN pos', ['ACL', 'Medial Meniscus', 'Effusion']].min() > .5
assert _s.loc['EN neg', ['ACL', 'Medial Meniscus', 'Effusion']].max() < .5
assert _s.loc['DE mix', 'Medial Meniscus'] > .5 and _s.loc['DE mix', 'Effusion'] < .5
assert _s.loc['FR mix', 'Effusion'] < .5 and _s.loc['FR mix', "Baker's"] > .5
assert _s.loc['NL pos', ['ACL', 'Medial Meniscus', "Baker's"]].min() > .5
print('rule_score self-test passed — clause-scoped negation and French elision both handled')
display(_s.style.background_gradient(cmap=CMAP_SEQ, vmin=0, vmax=1).format('{:.2f}'))

In [ ]:
if CFG.RUN_TEACHER:
    from sklearn.metrics import roc_auc_score, average_precision_score

    t0 = time.time()
    rules = pd.DataFrame([rule_score(x) for x in train['Report']], index=train.index)[TARGETS]
    print(f'rules computed in {time.time()-t0:.1f}s')

    ev, m = [], has_label & has_report
    for c in TARGETS:
        y, p = train.loc[m, c].values, rules.loc[m, c].values
        if len(np.unique(y)) < 2:
            continue
        ev.append({'label': c, 'n_pos': int(y.sum()),
                   'rule_auc': roc_auc_score(y, p), 'rule_ap': average_precision_score(y, p)})
    ev = pd.DataFrame(ev).sort_values('rule_auc')
    print('\nRule quality on the labelled subset:')
    display(ev.style.background_gradient(cmap=CMAP_SEQ, subset=['rule_auc'], vmin=.5, vmax=1)
              .format({'rule_auc': '{:.3f}', 'rule_ap': '{:.3f}'}))

Rules are the floor. The main teacher is character n-gram TF-IDF plus logistic regression, trained on
the labelled subset and scored out of fold. The rule outputs are appended as extra columns so the model
gets a hint on languages that are rare in the labelled part.

In [ ]:
if CFG.RUN_TEACHER:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from scipy import sparse

    txt = train['Report'].fillna('').str.lower().values
    fit_mask = (has_label & has_report).values

    vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5,
                               max_features=300_000, sublinear_tf=True)
    vec_word = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=3,
                               max_features=200_000, sublinear_tf=True)
    X = sparse.hstack([vec_char.fit_transform(txt),
                       vec_word.fit_transform(txt),
                       sparse.csr_matrix(rules.values.astype(np.float32))]).tocsr()
    print('feature matrix:', X.shape)

    Xfit = X[fit_mask]
    teacher = pd.DataFrame(index=train.index, columns=TARGETS, dtype=np.float32)
    oof_rows = []

    for c in TARGETS:
        y = train.loc[fit_mask, c].values.astype(int)
        if y.sum() < 10:
            teacher[c] = rules[c]
            continue
        oof = np.zeros(len(y))
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=CFG.seed).split(Xfit, y):
            lr = LogisticRegression(C=4.0, max_iter=2000, class_weight='balanced')
            lr.fit(Xfit[tr], y[tr])
            oof[va] = lr.predict_proba(Xfit[va])[:, 1]
        teacher[c] = LogisticRegression(C=4.0, max_iter=2000, class_weight='balanced') \
                        .fit(Xfit, y).predict_proba(X)[:, 1]
        oof_rows.append({'label': c, 'n_pos': int(y.sum()), 'text_auc': roc_auc_score(y, oof)})

    teach_ev = pd.DataFrame(oof_rows).merge(ev[['label', 'rule_auc']], on='label', how='left')
    teach_ev['gain'] = teach_ev.text_auc - teach_ev.rule_auc
    teach_ev = teach_ev.sort_values('text_auc')
    print(f'macro text AUC: {teach_ev.text_auc.mean():.4f}')

    fig, ax = plt.subplots(figsize=(9, 4.4))
    yy = np.arange(len(teach_ev))
    ax.hlines(yy, teach_ev.rule_auc, teach_ev.text_auc, color='#cfd4da', lw=2, zorder=1)
    ax.scatter(teach_ev.rule_auc, yy, s=55, color=ICE, label='rules only', zorder=2)
    ax.scatter(teach_ev.text_auc, yy, s=55, color=FIRE, label='rules + TF-IDF LR', zorder=2)
    ax.axvline(.85, color=MIDLINE, ls=':', lw=1, label='0.85 — below this, down-weight pseudo-labels')
    ax.set_yticks(yy); ax.set_yticklabels(teach_ev.label); ax.set_xlim(.45, 1.02)
    ax.set_xlabel('out-of-fold AUC, report -> label')
    ax.grid(axis='x'); ax.grid(axis='y', visible=False)
    ax.set_title('Teacher quality per target'); ax.legend(fontsize=8, loc='lower left')
    plt.tight_layout(); plt.show()
    display(teach_ev.round(3))

In [ ]:
if CFG.RUN_TEACHER:
    # final target table for the student: hard label where available, teacher output otherwise
    soft = teacher.copy()
    soft.loc[has_label, TARGETS] = train.loc[has_label, TARGETS].values
    soft.loc[~has_report & ~has_label, TARGETS] = np.nan        # unusable studies

    targets_df = pd.concat([train[['StudyInstanceUID']], soft], axis=1)
    targets_df['is_hard'] = has_label.astype(int)
    targets_df['w'] = np.where(has_label, 1.0, CFG.pseudo_w)
    targets_df = targets_df.dropna(subset=TARGETS)
    targets_df.to_csv(CFG.work_dir / 'targets.csv', index=False)

    print(f'training studies after the teacher: {len(targets_df):,} '
          f'(hard {targets_df.is_hard.sum():,} / soft {(1-targets_df.is_hard).sum():,})')

    cmp_df = pd.DataFrame({
        'hard_prevalence': train.loc[has_label, TARGETS].mean(),
        'soft_mean': targets_df.loc[targets_df.is_hard == 0, TARGETS].mean(),
    })
    fig, ax = plt.subplots(figsize=(9, 4))
    yy = np.arange(12)
    ax.barh(yy - .2, cmp_df.hard_prevalence, height=.38, color=ICE, label='hard prevalence')
    ax.barh(yy + .2, cmp_df.soft_mean, height=.38, color=FIRE, label='mean soft label')
    ax.set_yticks(yy); ax.set_yticklabels(cmp_df.index); ax.invert_yaxis()
    ax.set_xlabel('rate'); ax.set_title('Hard prevalence versus teacher output on unlabelled studies')
    ax.grid(axis='x'); ax.grid(axis='y', visible=False); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

> **Diagnostic:** a large gap between the two bars means either the labelled subset was enriched for
> abnormalities rather than sampled at random, or the teacher is biased. The first is more likely, and
> it is one more reason to validate the student on hard labels only.

# Part 4 — Validation

Three requirements:

1. **Multilabel stratification**, otherwise `Fracture` and `Synovitis` vanish from some folds.
2. **Grouping** by patient or duplicated report, against leakage.
3. **Scoring on hard labels only.** Measuring AUC against pseudo-labels measures agreement with the
   teacher, not with the truth.

In [ ]:
def multilabel_stratified_kfold(Y: np.ndarray, n_splits=5, seed=42) -> np.ndarray:
    """Greedy iterative stratification (Sechidis et al.), no external dependency."""
    rng = np.random.default_rng(seed)
    n, L = Y.shape
    fold_of = np.full(n, -1)
    desired = Y.sum(0)[None, :] / n_splits * np.ones((n_splits, 1))
    fold_count = np.zeros(n_splits)
    target_size = n / n_splits
    order = rng.permutation(n)
    order = order[np.argsort(Y[order].sum(1), kind='stable')[::-1]]     # label-rich rows first
    for i in order:
        lbl = np.where(Y[i] > 0)[0]
        if len(lbl) == 0:
            f = int(np.argmin(fold_count - target_size))
        else:
            rarest = lbl[np.argmin(Y[:, lbl].sum(0))]
            need = desired[:, rarest]
            cand = np.where(need == need.max())[0]
            f = int(cand[np.argmin(fold_count[cand])])
        fold_of[i] = f
        fold_count[f] += 1
        desired[f, lbl] -= 1
    return fold_of


# --- self-test: folds must be balanced in size and in rare positives ---
_rng = np.random.default_rng(0)
_Y = (_rng.random((2000, 12)) <
      np.array([.2, .1, .25, .12, .15, .08, .2, .3, .05, .09, .13, .02])).astype(int)
_f = multilabel_stratified_kfold(_Y, 5, 42)
_sizes = np.bincount(_f)
_rare = pd.DataFrame(_Y).groupby(_f).sum()[11]
assert _sizes.max() - _sizes.min() <= 1, _sizes
assert _rare.max() - _rare.min() <= 2, _rare.to_dict()
print(f'stratification self-test passed | fold sizes {_sizes.tolist()} | '
      f'rarest-label positives {_rare.tolist()}')

In [ ]:
if CFG.RUN_TEACHER:
    tdf = targets_df.copy()
    tdf['fold'] = multilabel_stratified_kfold((tdf[TARGETS].values > .5).astype(int),
                                              CFG.folds, CFG.seed)

    # group by duplicated report when PatientID is unavailable
    rep_hash = train.set_index('StudyInstanceUID')['Report'].fillna('').apply(
        lambda s: hashlib.md5(s.strip().encode()).hexdigest() if len(s.strip()) > 30 else None)
    tdf['grp'] = tdf.StudyInstanceUID.map(rep_hash)
    g = tdf.dropna(subset=['grp']).groupby('grp')['fold'].nunique()
    leaky = g[g > 1].index
    if len(leaky):
        first = tdf.dropna(subset=['grp']).groupby('grp')['fold'].first()
        tdf.loc[tdf.grp.isin(leaky), 'fold'] = tdf.loc[tdf.grp.isin(leaky), 'grp'].map(first)
        print(f'repaired {len(leaky)} groups that spanned several folds')

    tdf.drop(columns=['grp']).to_csv(CFG.work_dir / 'folds.csv', index=False)
    print('fold sizes:', tdf.fold.value_counts().sort_index().to_dict())

    hard_by_fold = tdf[tdf.is_hard == 1].groupby('fold')[TARGETS].sum().astype(int)
    fig, ax = plt.subplots(figsize=(11, 3.4))
    sns.heatmap(hard_by_fold, annot=True, fmt='d', cmap=CMAP_SEQ, linewidths=.6, linecolor='white',
                ax=ax, annot_kws={'size': 8}, cbar_kws={'shrink': .8})
    ax.set_title('Hard positives per fold — the real precision of your validation')
    plt.tight_layout(); plt.show()

# Part 5 — Student: EfficientNetV2 2.5D with attention pooling

```
(B, V, D, H, W) uint8
  └─ 2.5D: every slice d becomes channels (d-1, d, d+1)
      └─ EfficientNetV2-S, shared across views → (B·V·D, C)
          └─ + learned view embedding
              └─ attention pool over D  → (B, V, C)
                  └─ masked attention pool over V → (B, C)
                      └─ linear head → 12 logits
```

Why this shape:

- one shared backbone across views — fewer parameters, more images per parameter;
- attention over slices, because a finding occupies two or three slices out of sixteen and a mean
  washes it out (this is precisely what `feats.mean(dim=2)` costs in the public baseline);
- a mask over views, so a study without an axial series does not feed the pool with zeros;
- an auxiliary per-view head, which stabilises the early epochs.

**Augmentation:** shift, scale, rotation up to ±10°, brightness and contrast jitter.
**No horizontal flip** — it swaps medial and lateral.

In [ ]:
if CFG.RUN_TRAIN or CFG.RUN_INFER:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import timm

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'device: {DEVICE} | timm {timm.__version__} | torch {torch.__version__}')

    # ---- AMP compatibility across torch versions ----------------------------
    _NEW_AMP = hasattr(torch.amp, 'GradScaler')

    def make_scaler():
        return (torch.amp.GradScaler('cuda', enabled=CFG.amp) if _NEW_AMP
                else torch.cuda.amp.GradScaler(enabled=CFG.amp))

    def autocast():
        enabled = CFG.amp and torch.cuda.is_available()
        return (torch.amp.autocast('cuda', enabled=enabled) if _NEW_AMP
                else torch.cuda.amp.autocast(enabled=enabled))

    # ---- backbone name resolution -------------------------------------------
    def resolve_backbone(candidates, pretrained):
        """'efficientnetv2_s' is NOT a registered timm name — pick the first one that is."""
        available = set(timm.list_models())
        with_weights = set(timm.list_models(pretrained=True))
        for name in candidates:
            base = name.split('.')[0]
            if pretrained and name in with_weights:
                return name
            if not pretrained and (name in available or base in available):
                return name if name in available else base
        for name in candidates:                       # last resort: drop the weight tag
            if name.split('.')[0] in available:
                print(f'[!] falling back to {name.split(".")[0]} without a specific weight tag')
                return name.split('.')[0]
        raise RuntimeError(f'None of {candidates} exists in timm {timm.__version__}')

    BACKBONE = resolve_backbone(CFG.backbone, CFG.pretrained)
    print('backbone resolved to:', BACKBONE)

In [ ]:
if CFG.RUN_TRAIN or CFG.RUN_INFER:

    class KneeDataset(Dataset):
        """Reads pre-cached npz volumes. Missing studies yield zeros with a zero mask."""

        def __init__(self, df, cache_dirs, train_mode=True, targets=None):
            self.df = df.reset_index(drop=True)
            self.train_mode = train_mode
            self.targets = targets
            self.index = {}                       # study -> path, built once, not per item
            for d in cache_dirs:
                d = Path(d)
                if d.exists():
                    for p in d.glob('*.npz'):
                        self.index.setdefault(p.stem, p)
            present = sum(s in self.index for s in self.df.StudyInstanceUID)
            print(f'  cache index: {len(self.index):,} studies | '
                  f'{present:,} of {len(self.df):,} rows resolved')

        def __len__(self):
            return len(self.df)

        def _augment(self, vol):
            """vol (V, D, H, W) uint8 -> float32 with geometric and photometric jitter. No flips."""
            V, D, H, W = vol.shape
            out = np.empty_like(vol)
            for v in range(V):
                ang = np.random.uniform(-10, 10)
                sc = np.random.uniform(0.9, 1.1)
                tx, ty = np.random.uniform(-.06, .06, 2) * W
                M = cv2.getRotationMatrix2D((W / 2, H / 2), ang, sc)
                M[0, 2] += tx; M[1, 2] += ty
                for d in range(D):
                    out[v, d] = cv2.warpAffine(vol[v, d], M, (W, H), borderMode=cv2.BORDER_CONSTANT)
            x = out.astype(np.float32) / 255.
            return np.clip(x * np.random.uniform(.85, 1.15) + np.random.uniform(-.08, .08), 0, 1)

        def __getitem__(self, i):
            r = self.df.iloc[i]
            p = self.index.get(r.StudyInstanceUID)
            if p is None:
                vol = np.zeros((CFG.N_VIEWS, CFG.N_SLICES, CFG.IMG_SIZE, CFG.IMG_SIZE), np.uint8)
                mask = np.zeros(CFG.N_VIEWS, np.float32)
            else:
                z = np.load(p)
                vol, mask = z['vol'], z['mask'].astype(np.float32)
            x = self._augment(vol) if self.train_mode else vol.astype(np.float32) / 255.
            item = {'x': torch.from_numpy(np.ascontiguousarray(x)),
                    'mask': torch.from_numpy(mask)}
            if self.targets is not None:
                item['y'] = torch.tensor(r[self.targets].values.astype(np.float32))
                item['w'] = torch.tensor(float(r.get('w', 1.0)))
            return item


    class AttnPool(nn.Module):
        def __init__(self, dim, hidden=128):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

        def forward(self, x, mask=None):                    # x: (B, N, C)
            a = self.net(x).squeeze(-1)
            if mask is not None:
                a = a.masked_fill(mask < .5, -1e4)          # -1e4, not -inf: fp16 safe
            a = a.softmax(-1)
            return (x * a.unsqueeze(-1)).sum(1), a


    class KneeNet(nn.Module):
        def __init__(self, backbone=None, n_views=CFG.N_VIEWS, n_out=12,
                     pretrained=CFG.pretrained, drop=CFG.drop_rate, grad_ckpt=CFG.grad_ckpt):
            super().__init__()
            self.enc = timm.create_model(backbone or BACKBONE, pretrained=pretrained,
                                         num_classes=0, in_chans=3, drop_rate=drop)
            if grad_ckpt:
                try:
                    self.enc.set_grad_checkpointing(True)
                except Exception:
                    pass
            C = self.enc.num_features
            self.view_emb = nn.Parameter(torch.zeros(n_views, C))
            nn.init.normal_(self.view_emb, std=.02)
            self.slice_pool = AttnPool(C)
            self.view_pool = AttnPool(C)
            self.drop = nn.Dropout(drop)
            self.head = nn.Linear(C, n_out)
            self.aux_head = nn.Linear(C, n_out)

        @staticmethod
        def to_25d(x):                                      # (B,V,D,H,W) -> (B,V,D,3,H,W)
            prev = torch.cat([x[:, :, :1], x[:, :, :-1]], 2)
            nxt = torch.cat([x[:, :, 1:], x[:, :, -1:]], 2)
            return torch.stack([prev, x, nxt], 3)

        def forward(self, x, mask):
            B, V, D, H, W = x.shape
            f = self.enc(self.to_25d(x).reshape(B * V * D, 3, H, W)).reshape(B, V, D, -1)
            f = f + self.view_emb[None, :, None, :]
            fv, _ = self.slice_pool(f.reshape(B * V, D, -1))
            fv = fv.reshape(B, V, -1)
            aux = self.aux_head(self.drop(fv))               # (B, V, 12)
            fs, w_view = self.view_pool(fv, mask)            # (B, C)
            return self.head(self.drop(fs)), aux, w_view

    print('dataset and model defined')

### Model smoke test

Build the network, push a random batch of the exact training shape through it, and run one backward
pass. This catches an unknown backbone name, a shape mismatch or an out-of-memory in seconds, instead
of twenty minutes into the first epoch. The masked view is checked explicitly: its attention weight
must be zero.

In [ ]:
if CFG.RUN_TRAIN or CFG.RUN_INFER:
    _m = KneeNet(pretrained=False).to(DEVICE)
    print(f'parameters: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M '
          f'| feature dim: {_m.enc.num_features}')

    _x = torch.rand(CFG.batch_size, CFG.N_VIEWS, CFG.N_SLICES,
                    CFG.IMG_SIZE, CFG.IMG_SIZE, device=DEVICE)
    _msk = torch.tensor([[1., 1., 0.]] * CFG.batch_size, device=DEVICE)   # axial missing on purpose
    _y = (torch.rand(CFG.batch_size, 12, device=DEVICE) > .7).float()

    _t0 = time.time()
    with autocast():
        _logit, _aux, _wv = _m(_x, _msk)
        _loss = F.binary_cross_entropy_with_logits(_logit, _y)
    _loss.backward()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

    assert _logit.shape == (CFG.batch_size, 12), _logit.shape
    assert _aux.shape == (CFG.batch_size, CFG.N_VIEWS, 12), _aux.shape
    assert float(_wv[:, 2].abs().max()) < 1e-3, 'a masked view still receives attention weight'
    print(f'forward + backward ok in {time.time()-_t0:.2f}s | logits {tuple(_logit.shape)} '
          f'| view weights {_wv[0].detach().float().cpu().numpy().round(3)}')
    if DEVICE == 'cuda':
        print(f'peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.2f} GB of '
              f'{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB')
        print('If that is close to the limit: lower CFG.batch_size or CFG.N_SLICES, raise CFG.accum.')

    del _m, _x, _msk, _y, _logit, _aux, _wv, _loss
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
if CFG.RUN_TRAIN:
    from sklearn.metrics import roc_auc_score

    def macro_auc(y, p):
        aucs = {}
        for i, c in enumerate(TARGETS):
            yi = (y[:, i] > .5).astype(int)
            aucs[c] = np.nan if yi.min() == yi.max() else roc_auc_score(yi, p[:, i])
        return float(np.nanmean(list(aucs.values()))), aucs


    def train_fold(fold, folds_df, cache_dirs):
        tr = folds_df[folds_df.fold != fold]
        va = folds_df[(folds_df.fold == fold) & (folds_df.is_hard == 1)]   # hard labels only
        print(f'\n=== fold {fold} | train {len(tr):,} | val (hard) {len(va):,} ===')

        dl_tr = DataLoader(KneeDataset(tr, cache_dirs, True, TARGETS), batch_size=CFG.batch_size,
                           shuffle=True, num_workers=CFG.num_workers, pin_memory=True,
                           drop_last=True, persistent_workers=CFG.num_workers > 0)
        dl_va = DataLoader(KneeDataset(va, cache_dirs, False, TARGETS), batch_size=CFG.batch_size,
                           shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

        model = KneeNet().to(DEVICE)
        bb = [p for n, p in model.named_parameters() if n.startswith('enc.')]
        hd = [p for n, p in model.named_parameters() if not n.startswith('enc.')]
        opt = torch.optim.AdamW([{'params': bb, 'lr': CFG.backbone_lr},
                                 {'params': hd, 'lr': CFG.lr}], weight_decay=CFG.wd)
        steps = max(1, len(dl_tr) // CFG.accum) * CFG.epochs
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[CFG.backbone_lr, CFG.lr],
                                                    total_steps=steps, pct_start=.1)
        scaler = make_scaler()

        hist, best = [], -1
        for ep in range(CFG.epochs):
            model.train(); t0, run, nb = time.time(), 0., 0
            opt.zero_grad(set_to_none=True)
            for it, b in enumerate(dl_tr):
                x, m = b['x'].to(DEVICE, non_blocking=True), b['mask'].to(DEVICE)
                y, w = b['y'].to(DEVICE), b['w'].to(DEVICE)
                with autocast():
                    logit, aux, _ = model(x, m)
                    l_main = F.binary_cross_entropy_with_logits(logit, y, reduction='none').mean(1)
                    l_aux = F.binary_cross_entropy_with_logits(
                        aux, y[:, None, :].expand_as(aux), reduction='none').mean(-1)
                    l_aux = (l_aux * m).sum(1) / m.sum(1).clamp(min=1)
                    loss = ((l_main + CFG.aux_w * l_aux) * w).mean() / CFG.accum
                scaler.scale(loss).backward()
                if (it + 1) % CFG.accum == 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                    opt.zero_grad(set_to_none=True)
                    if sched.last_epoch < steps - 1:
                        sched.step()
                run += loss.item() * CFG.accum; nb += 1

            model.eval(); P, Y = [], []
            with torch.no_grad(), autocast():
                for b in dl_va:
                    logit, _, _ = model(b['x'].to(DEVICE), b['mask'].to(DEVICE))
                    P.append(logit.float().sigmoid().cpu().numpy()); Y.append(b['y'].numpy())
            P, Y = np.concatenate(P), np.concatenate(Y)
            auc, per = macro_auc(Y, P)
            hist.append({'epoch': ep, 'loss': run / max(nb, 1), 'auc': auc, **per})
            print(f'  epoch {ep} | loss {run/max(nb,1):.4f} | macro AUC {auc:.4f} '
                  f'| {time.time()-t0:.0f}s', flush=True)
            if auc > best:
                best = auc
                torch.save(model.state_dict(), CFG.work_dir / f'model_f{fold}.pt')

        hist = pd.DataFrame(hist)
        fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
        axes[0].plot(hist.epoch, hist.loss, marker='o', color=ICE)
        axes[0].set_title(f'fold {fold} — training loss'); axes[0].set_xlabel('epoch')
        axes[1].plot(hist.epoch, hist.auc, marker='o', color=FIRE)
        axes[1].axhline(best, ls='--', lw=1, color=MIDLINE, label=f'best {best:.4f}')
        axes[1].set_title(f'fold {fold} — validation macro AUC')
        axes[1].set_xlabel('epoch'); axes[1].legend(fontsize=8)
        plt.tight_layout(); plt.show()

        best_row = hist.loc[hist.auc.idxmax(), TARGETS].astype(float).sort_values()
        fig, ax = plt.subplots(figsize=(9, 3.6))
        ax.barh(best_row.index, best_row.values, color=grad(12), edgecolor='white', height=.7)
        ax.axvline(.5, color=MIDLINE, ls=':', lw=1)
        ax.set_xlim(.4, 1.0); ax.set_title(f'fold {fold} — per-target AUC at the best epoch')
        ax.grid(axis='x'); ax.grid(axis='y', visible=False)
        plt.tight_layout(); plt.show()

        del model; gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        return best


    folds_df = pd.read_csv(CFG.work_dir / 'folds.csv')
    cache_dirs = [CFG.cache_dir, CFG.cache_dir / f'shard{CFG.SHARD_ID}'] + list(CFG.extra_cache)
    scores = [train_fold(f, folds_df, cache_dirs) for f in CFG.train_folds]
    print('\nCV:', np.round(scores, 4), '| mean', round(float(np.mean(scores)), 4))

# Part 6 — Inference and submission

The submission notebook has no internet: `pretrained=False`, weights come from your own dataset, and
DICOM is decoded on the fly by the very same functions — the preprocessing must be **bit-identical** to
training, which is exactly why `load_series_volume` is shared rather than reimplemented here.

About 1300 studies × 3 views × 16 slices ≈ 62k images. On a T4 with EfficientNetV2-S and AMP that is
roughly 30–45 minutes including DICOM decoding, comfortably inside the 9-hour limit.

Fold predictions are combined by **rank**, not by probability: the metric is rank-based and separate
folds are not calibrated against each other.

In [ ]:
if CFG.RUN_INFER:
    class TestDataset(Dataset):
        """Decodes DICOM on the fly — no cache needed for ~1300 studies."""

        def __init__(self, manifest_df, root):
            self.df = manifest_df.reset_index(drop=True)
            self.root = Path(root)

        def __len__(self):
            return len(self.df)

        def __getitem__(self, i):
            r = self.df.iloc[i]
            vol = np.zeros((CFG.N_VIEWS, CFG.N_SLICES, CFG.IMG_SIZE, CFG.IMG_SIZE), np.uint8)
            mask = np.zeros(CFG.N_VIEWS, np.float32)
            for vi, slot in enumerate(CFG.VIEWS):
                sid = r.get(slot)
                if not isinstance(sid, str):
                    continue
                try:
                    v = load_series_volume(self.root / r.StudyInstanceUID / sid, VIEW_SPEC[slot][0])
                except Exception:
                    v = None
                if v is not None:
                    vol[vi], mask[vi] = v, 1
            return {'x': torch.from_numpy(vol.astype(np.float32) / 255.),
                    'mask': torch.from_numpy(mask), 'idx': i}

    test_manifest = test.merge(build_manifest(test_series, CFG.comp_dir / 'test_series'),
                               on='StudyInstanceUID', how='left')
    print('test manifest:', test_manifest.shape, '| studies with no usable view:',
          int(test_manifest[list(VIEW_SPEC)].isna().all(axis=1).sum()))

    dl = DataLoader(TestDataset(test_manifest, CFG.comp_dir / 'test_series'),
                    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

    ckpts = sorted(CFG.work_dir.glob('model_f*.pt')) or \
            sorted(Path('/kaggle/input').glob('*/model_f*.pt'))
    print('checkpoints found:', len(ckpts), [c.name for c in ckpts])

    ranked = np.zeros((len(test_manifest), 12), np.float32)
    for ck in ckpts:
        model = KneeNet(pretrained=False).to(DEVICE)
        model.load_state_dict(torch.load(ck, map_location=DEVICE))
        model.eval()
        fold_pred = np.zeros_like(ranked)
        with torch.no_grad(), autocast():
            for b in dl:
                logit, _, _ = model(b['x'].to(DEVICE), b['mask'].to(DEVICE))
                fold_pred[b['idx'].numpy()] = logit.float().sigmoid().cpu().numpy()
        ranked += pd.DataFrame(fold_pred).rank(pct=True).values.astype(np.float32)
        del model; gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    ranked /= max(1, len(ckpts))

    sub = pd.DataFrame(ranked, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', test_manifest.StudyInstanceUID.values)
    sub = sample_sub[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub = sub[sample_sub.columns.tolist()]
else:
    sub = sample_sub.copy()                 # always emit a valid file
    print('RUN_INFER is off — writing the sample submission')

sub.to_csv('submission.csv', index=False)
assert list(sub.columns) == list(sample_sub.columns), 'column order must match sample_submission'
assert len(sub) == len(sample_sub) and int(sub[TARGETS].isna().sum().sum()) == 0
print('submission.csv written:', sub.shape)
display(sub.head())

# Part 7 — Where to go from here

Ordered by gain per hour of work.

**1. Improve the teacher first — cheap, and it raises the ceiling for everything else.**
Pseudo-labels are the upper bound on the student. Check the per-target out-of-fold AUC from Part 3; for
the weak ones (`Synovitis` and `Lateral OA` are the usual suspects) add lexicon terms and inspect the
heaviest n-grams of the logistic regression — they show exactly which words are missing. A multilingual
encoder such as `xlm-roberta-base`, fine-tuned on the labelled subset, typically adds another 0.02–0.05
AUC over TF-IDF on long reports.

**2. Verify that laterality normalisation actually worked.**
Diagnostic: train, then compare the AUC of `Medial OA` against `Lateral OA`. If both sit near 0.6 while
`Effusion` is high, the sides are still mixed and the model cannot separate the compartments. Confirm on
a handful of coronal slices by eye.

**3. Resolution and slice count.** 224 × 16 is a compromise. Menisci and cartilage are small structures;
moving to 320 × 32 usually helps the meniscus and OA targets noticeably, at a proportional cost in time.

**4. More series per view.** One series per slot right now. A multiple-instance variant — every series
of a study, with attention across series — removes the selection heuristic and brings T1/PD information
back in.

**5. Target-specific treatment.** `Fracture` and `Baker's` are local and rare; a dedicated head with
aggressive positive oversampling helps more than a larger backbone.

**6. Ensembling.** Different backbones (EfficientNetV2-S, ConvNeXt-T, MaxViT-T) and different seeds,
combined by rank. On macro-AUC this reliably adds 0.01–0.02.

**7. The efficiency track.** The metric charges you per second. Recipe: `tf_efficientnetv2_b0` or
`resnet34`, 160–192 px, 12 slices, two views instead of three (sagittal plus coronal), fp16, one fold,
a larger batch. That usually costs 0.01–0.015 macro AUC for a 3–4× speed-up, a favourable trade under
the efficiency formula.

**What not to do**

- Image plus text fusion at inference — there are no reports in the test set.
- Horizontal flip in augmentation or TTA — it destroys the medial/lateral distinction.
- Validating on pseudo-labels — that measures agreement with the teacher, not accuracy.
- `pos_weight` of 100 on rare labels — unstable gradients, and macro-AUC ignores the prior anyway.
- Probability calibration or threshold tuning — the metric is rank-based, so it changes nothing.